In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import *
from pyspark.sql.types import *

### **Data Reading**

In [0]:
%python
df=spark.read.format('parquet')\
    .option('inferSchema',True)\
    .load('abfss://bronze@carsankidatalake.dfs.core.windows.net/rawdata')    

# **Data Transformation**

In [0]:
df=df.withColumn('Model_category',split(col('Model_ID'),'-')[0])

In [0]:
df.withColumn('Units_Sold',col('Units_Sold').cast(StringType()))

DataFrame[Branch_ID: string, Dealer_ID: string, Model_ID: string, Revenue: bigint, Units_Sold: string, Date_ID: string, Day: int, Month: int, Year: int, BranchName: string, DealerName: string, Model_category: string]

In [0]:
df=df.withColumn('RevPerUnit',col('Revenue')/col('Units_Sold'))

In [0]:
df.write.format('parquet')\
    .mode('overwrite')\
    .option('path','abfss://silver@carsankidatalake.dfs.core.windows.net/carsales')   \
    .save()    

# **Quering Silver Data**

In [0]:
%sql
Select count(*) from parquet.`abfss://silver@carsankidatalake.dfs.core.windows.net/carsales`

count(*)
1853


In [0]:
display(df.groupBy(col("Year"),col("BranchName")).agg(sum(col("Units_Sold")).alias("Total_Units")).orderBy(col("Year").asc(),col("Total_Units").desc()))

Year,BranchName,Total_Units
2017,Alpine Motors,24
2017,Bristol Motors,23
2017,BMW Motors,23
2017,Aston Martin Motors,23
2017,Acura Motors,23
2017,Ariel Motors,21
2017,Daihatsu Motors,21
2017,Gilbern Motors,21
2017,Asia Motors Motors,20
2017,DAF Motors,19


Databricks visualization. Run in Databricks to view.